In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

from src.potentials import barrier
from src.split_operator import evolve

In [ ]:
# Grid parameters
n_qubits = 8
N = 2**n_qubits
L = 40.0
dx = L / N
x = np.linspace(-L/2, L/2, N, endpoint=False)

# Momentum grid (DFT order)
dp = 2 * np.pi / L
p = np.fft.fftfreq(N) * N * dp

# Physical parameters
omega = 1.0
m = 1.0

# Barrier parameters
V0 = 3.0
a = 0.5

# Initial wavepacket parameters
x0 = -5.0
sigma = 1.0
p0 = 2.0

In [ ]:
# Initial wavepacket
psi = np.exp(-(x - x0)**2 / (4 * sigma**2)) * np.exp(1j * p0 * x)
psi = psi / np.sqrt(np.sum(np.abs(psi)**2) * dx)

# Plot initial state with barrier
V = barrier(x, V0, a)

plt.figure(figsize=(10, 4))
plt.plot(x, np.abs(psi)**2, label=r'$|\psi(x)|^2$')
plt.plot(x, V / V0 * np.max(np.abs(psi)**2), label=f'Barrier (scaled)', color='red')
plt.xlabel("x")
plt.ylabel(r'$|\psi(x)|^2$')
plt.title("Initial wavepacket and barrier")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Time parameters
dt = 0.05
n_steps = 200  # total time = 10, enough for the packet to cross the barrier

# Classical evolution
psi_sq_classical, x_means_classical = evolve(psi, V, x, p, dt, n_steps, method='classical')

# Quantum evolution
psi_sq_quantum, x_means_quantum = evolve(psi, V, x, p, dt, n_steps, method='quantum')

In [ ]:
# Visualization of classical evolution
t = np.arange(n_steps) * dt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: expectation value of position
axes[0].plot(t, x_means_classical, label='Classical Trotter')
axes[0].plot(t, x_means_quantum, '--', label='Qiskit')
axes[0].axhline(-a, color='red', linestyle=':', label='Barrier')
axes[0].axhline(a, color='red', linestyle=':')
axes[0].set_xlabel("t")
axes[0].set_ylabel("<x>(t)")
axes[0].set_title("Expectation value of position")
axes[0].legend()
axes[0].grid(True)

# Right: probability density as 2D image
positions_array = np.array(psi_sq_quantum)
im = axes[1].imshow(positions_array, aspect='auto', 
                     extent=[-L/2, L/2, n_steps*dt, 0],
                     cmap='inferno')
axes[1].axvline(-a, color='white', linestyle='--', linewidth=1, label='Barrier')
axes[1].axvline(a, color='white', linestyle='--', linewidth=1)
axes[1].set_xlabel("x")
axes[1].set_ylabel("t")
axes[1].set_title("Probability density |psi(x,t)|^2 (Qiskit)")
axes[1].legend()
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()